# 262. Trips and Users

**Difficulty:** Hard &nbsp;|&nbsp; **Topics:** database, join, group-by, conditional-aggregation
&nbsp;|&nbsp; [LeetCode](https://leetcode.com/problems/trips-and-users/)

```
Table: Trips                             Table: Users
+-------------+----------+               +-------------+----------+
| Column Name | Type     |               | Column Name | Type     |
+-------------+----------+               +-------------+----------+
| id          | int      |               | users_id    | int      |
| client_id   | int      |               | banned      | enum     |
| driver_id   | int      |               | role        | enum     |
| city_id     | int      |               +-------------+----------+
| status      | enum     |               users_id is the primary key.
| request_at  | varchar  |               banned is 'Yes' or 'No'.
+-------------+----------+               role is 'client', 'driver' or 'partner'.
id is the primary key.
status is 'completed', 'cancelled_by_driver' or 'cancelled_by_client'.
client_id and driver_id both reference Users.users_id.
```

The **cancellation rate** is computed by dividing the number of cancelled (by client or
driver) requests with **unbanned users** by the total number of requests with unbanned
users on that day.

Write a solution to find the cancellation rate of requests with unbanned users (**both
client and driver must not be banned**) each day between `2013-10-01` and `2013-10-03`.
Round the cancellation rate to **two decimal places**.

Return the result table in **any order**. The result columns must be called `Day` and
`Cancellation Rate`.

---

### Example

```
Trips:
+----+-----------+-----------+---------+---------------------+------------+
| id | client_id | driver_id | city_id | status              | request_at |
+----+-----------+-----------+---------+---------------------+------------+
| 1  | 1         | 10        | 1       | completed           | 2013-10-01 |
| 2  | 2         | 11        | 1       | cancelled_by_driver | 2013-10-01 |
| 3  | 3         | 12        | 6       | completed           | 2013-10-01 |
| 4  | 4         | 13        | 6       | cancelled_by_client | 2013-10-01 |
| 5  | 1         | 10        | 1       | completed           | 2013-10-02 |
| 6  | 2         | 11        | 6       | completed           | 2013-10-02 |
| 7  | 3         | 12        | 6       | completed           | 2013-10-02 |
| 8  | 2         | 12        | 12      | completed           | 2013-10-03 |
| 9  | 3         | 10        | 12      | completed           | 2013-10-03 |
| 10 | 4         | 13        | 12      | cancelled_by_driver | 2013-10-03 |
+----+-----------+-----------+---------+---------------------+------------+

Users: user 2 is a banned client; everyone else is unbanned.

Output:
+------------+-------------------+
| Day        | Cancellation Rate |
+------------+-------------------+
| 2013-10-01 | 0.33              |
| 2013-10-02 | 0.00              |
| 2013-10-03 | 0.50              |
+------------+-------------------+
```

On 2013-10-01 there are 4 trips, but trip 2 involves banned user 2, so it is excluded
entirely: 3 trips remain, 1 was cancelled, `1/3 = 0.33`.

---

The hardest problem in this folder, and not because of any one technique - because
there are **four** separate requirements and each one is a chance to be quietly wrong.
Read the statement three times before you write anything.

## Before you write anything

**1.** List the four requirements as separate sentences before you write a line of SQL.
They are: which **trips** to exclude, which **dates** to keep, what counts as
**cancelled**, and what to do about the **rounding**. Write them out - most wrong answers
here are a correct query that forgot one of the four.

**2.** **Banned users.** A trip is excluded if the client is banned **or** the driver is
banned. Note that `Users` holds both, distinguished by `role`. So you must check
`client_id` and `driver_id` against the *same* table, twice. Which technique from #181
does that need? Write it, then say what would go wrong if you only checked the client.

**3.** Careful with the join direction. If you `JOIN Users u ON u.users_id = t.client_id`,
what happens to a trip whose client is not in `Users` at all? Is that possible under the
schema? Decide whether an inner join or a `NOT IN`-style exclusion expresses the
requirement more faithfully, and be able to defend it.

**4.** **Cancelled** means `cancelled_by_driver` **or** `cancelled_by_client` - not just
one. The neat way to count them is **conditional aggregation**:

```sql
SUM(CASE WHEN status <> 'completed' THEN 1 ELSE 0 END)
```

Say why `SUM` of a `CASE` counts things, and why `COUNT(CASE WHEN ... THEN 1 END)`
works too but `COUNT(CASE WHEN ... THEN 1 ELSE 0 END)` does **not**. (The reason is
`COUNT` ignoring `NULL` - and it is a classic interview follow-up.)

**5.** **Division.** `1 / 3` in SQL with two integers may be `0`, not `0.33`. Check what
SQLite does, then write the division so it is a real number regardless - multiply by
`1.0`, or cast. Then apply `ROUND(x, 2)`. Getting `0` for every row is the single most
common wrong answer to this problem.

**6.** Days with **no** unbanned trips at all: should they appear with rate `0.00`, or
not appear? Work out what a `GROUP BY` over the filtered rows does naturally, and check
it against the statement. (A `GROUP BY` cannot invent a day that has no rows - so what
does that imply?)

**7.** The output column is called `Cancellation Rate`, **with a space**. That needs
quoting: `AS "Cancellation Rate"` in SQLite and standard SQL, or backticks in MySQL.

## Two routes

**A - join twice, then conditional aggregation** *(write this first)*

```sql
SELECT t.request_at AS Day,
       ROUND(SUM(CASE WHEN t.status <> 'completed' THEN 1.0 ELSE 0 END) / COUNT(*), 2)
           AS "Cancellation Rate"
FROM Trips t
JOIN Users c ON c.users_id = t.client_id AND c.banned = 'No'
JOIN Users d ON d.users_id = t.driver_id AND d.banned = 'No'
WHERE t.request_at BETWEEN '2013-10-01' AND '2013-10-03'
GROUP BY t.request_at
```

Two joins to the same `Users` table, aliased `c` and `d` - #181's self-join idea applied
to a *different* table twice. Putting `banned = 'No'` in the `ON` makes the join itself
do the filtering, so any trip with a banned participant simply never appears.

`COUNT(*)` after those joins is "unbanned trips that day"; the `SUM(CASE ...)` is
"cancelled ones". The `1.0` is question 5's fix - it forces the whole sum to be a real
number, so the division is not integer division.

Dates are stored as `'YYYY-MM-DD'` text, which sorts and compares correctly as strings -
which is exactly why that format is worth using everywhere.

**B - exclude with `NOT IN`**

```sql
WHERE t.client_id NOT IN (SELECT users_id FROM Users WHERE banned = 'Yes')
  AND t.driver_id NOT IN (SELECT users_id FROM Users WHERE banned = 'Yes')
```

Reads closer to the sentence "neither participant is banned". Note that it is **safe**
here, unlike #183 - `users_id` is a primary key and cannot be null, so the trap does not
apply. Say why out loud, because "I know when this construct is dangerous and this is not
one of those times" is a better place to be than avoiding it forever.

> **Four requirements, four chances to be wrong.** This problem is not hard because any
> one part is hard - it is hard because it is the first one where you have to hold the
> whole statement in your head at once. Write the four sentences down first, then check
> your finished query against them one at a time. That habit is what the problem is
> actually teaching.

In [ ]:
SOLUTION = '''
'''

### The test harness

Every notebook in this folder runs your SQL for real, against a fresh **SQLite**
database built from scratch for each test case. Nothing is mocked and nothing is
pattern-matched - if your query runs and returns the right rows, it passes.

`check(name, data, expected)` creates the tables, inserts that case's rows, executes
whatever string is in `SOLUTION`, and compares. It checks two things: the **rows**
(as a set - row order does not matter unless the problem says it does) and the
**column names**, because a query that returns the right numbers under the wrong
headings is not the answer the question asked for.

On failure it prints your rows next to the expected ones and names which rows are
missing and which should not be there.

`show(name, data, query)` is there for you: run *any* query against any dataset and
print it. Use it to look at intermediate results while you are working - especially
to run the deliberately-wrong version of your query and watch what it does.

> **SQLite here, MySQL on LeetCode.** They agree on everything these problems need -
> joins, `GROUP BY`/`HAVING`, subqueries, `LIMIT`/`OFFSET`, `COALESCE`, and window
> functions like `DENSE_RANK`. Where a problem needs something MySQL does differently,
> the notebook says so in the routes section. Write standard SQL and both will take it.

Run this cell; don't edit it.

In [ ]:
import sqlite3

SCHEMA = """CREATE TABLE Trips (id INTEGER, client_id INTEGER, driver_id INTEGER, city_id INTEGER,
                    status TEXT, request_at TEXT);
CREATE TABLE Users (users_id INTEGER, banned TEXT, role TEXT);"""

EXPECTED_COLUMNS = ['Day', 'Cancellation Rate']
ORDERED = False


def _norm(rows):
    return rows if ORDERED else sorted(rows, key=lambda r: tuple((v is None, str(v)) for v in r))


def check(name, data_sql, expected):
    """Build a fresh in-memory database, run SOLUTION against it, compare."""
    con = sqlite3.connect(":memory:")
    try:
        con.executescript(SCHEMA)
        if data_sql.strip():
            con.executescript(data_sql)
    except sqlite3.Error as e:
        print(f"FAIL {name}")
        print(f"       the harness could not build the tables: {e}")
        return False

    if not SOLUTION.strip():
        print(f"FAIL {name}")
        print("       SOLUTION is empty - write your query in the cell above")
        return False

    try:
        cur = con.execute(SOLUTION)
        got = [tuple(r) for r in cur.fetchall()]
        cols = [d[0] for d in cur.description] if cur.description else []
    except sqlite3.Error as e:
        print(f"FAIL {name}")
        print(f"       your query raised {type(e).__name__}: {e}")
        return False

    cols_ok = [c.lower() for c in cols] == [c.lower() for c in EXPECTED_COLUMNS]
    rows_ok = _norm(got) == _norm(expected)

    if cols_ok and rows_ok:
        print(f"OK   {name}")
        return True

    print(f"FAIL {name}")
    if not cols_ok:
        print(f"       column names  {cols}")
        print(f"       should be     {EXPECTED_COLUMNS}")
    if not rows_ok:
        missing = [r for r in expected if r not in got]
        extra = [r for r in got if r not in expected]
        print(f"       you returned {len(got)} row(s), expected {len(expected)}"
              + ("   (row order matters here)" if ORDERED else "   (row order does not matter)"))
        for r in got[:6]:
            print(f"         got       {r}")
        for r in expected[:6]:
            print(f"         expected  {r}")
        if missing:
            print(f"       rows you are MISSING: {missing[:4]}")
        if extra:
            print(f"       rows you should NOT have: {extra[:4]}")
    return False


def show(name, data_sql, query):
    """Run any query against a dataset and print it - for exploring, not for grading."""
    con = sqlite3.connect(":memory:")
    con.executescript(SCHEMA)
    if data_sql.strip():
        con.executescript(data_sql)
    cur = con.execute(query)
    cols = [d[0] for d in cur.description]
    rows = cur.fetchall()
    print(f"-- {name}")
    print("   " + " | ".join(str(c) for c in cols))
    for r in rows:
        print("   " + " | ".join("NULL" if v is None else str(v) for v in r))
    if not rows:
        print("   (no rows)")

In [ ]:
# tests
LEETCODE_USERS = '''
INSERT INTO Users VALUES (1,'No','client'),(2,'Yes','client'),(3,'No','client'),
                         (4,'No','client'),(10,'No','driver'),(11,'No','driver'),
                         (12,'No','driver'),(13,'No','driver');
'''

check("the LeetCode example", LEETCODE_USERS + '''
INSERT INTO Trips VALUES
 (1,1,10,1,'completed','2013-10-01'),
 (2,2,11,1,'cancelled_by_driver','2013-10-01'),
 (3,3,12,6,'completed','2013-10-01'),
 (4,4,13,6,'cancelled_by_client','2013-10-01'),
 (5,1,10,1,'completed','2013-10-02'),
 (6,2,11,6,'completed','2013-10-02'),
 (7,3,12,6,'completed','2013-10-02'),
 (8,2,12,12,'completed','2013-10-03'),
 (9,3,10,12,'completed','2013-10-03'),
 (10,4,13,12,'cancelled_by_driver','2013-10-03');
''', [('2013-10-01', 0.33), ('2013-10-02', 0.0), ('2013-10-03', 0.5)])

check("question 5: everything cancelled - the rate is 1.0, not 1", '''
INSERT INTO Users VALUES (1,'No','client'),(10,'No','driver');
INSERT INTO Trips VALUES (1,1,10,1,'cancelled_by_client','2013-10-01');
''', [('2013-10-01', 1.0)])

check("question 5: nothing cancelled - 0.0, not 0", '''
INSERT INTO Users VALUES (1,'No','client'),(10,'No','driver');
INSERT INTO Trips VALUES (1,1,10,1,'completed','2013-10-01');
''', [('2013-10-01', 0.0)])

check("*** question 2: a banned DRIVER excludes the trip too ***", '''
INSERT INTO Users VALUES (1,'No','client'),(10,'Yes','driver'),(11,'No','driver');
INSERT INTO Trips VALUES (1,1,10,1,'cancelled_by_driver','2013-10-01'),
                         (2,1,11,1,'completed','2013-10-01');
''', [('2013-10-01', 0.0)])

check("*** question 6: a day where EVERY trip is banned does not appear ***", '''
INSERT INTO Users VALUES (1,'Yes','client'),(10,'No','driver'),(2,'No','client');
INSERT INTO Trips VALUES (1,1,10,1,'completed','2013-10-01'),
                         (2,2,10,1,'completed','2013-10-02');
''', [('2013-10-02', 0.0)])

check("*** dates outside the window are excluded ***", '''
INSERT INTO Users VALUES (1,'No','client'),(10,'No','driver');
INSERT INTO Trips VALUES (1,1,10,1,'cancelled_by_client','2013-09-30'),
                         (2,1,10,1,'completed','2013-10-02'),
                         (3,1,10,1,'cancelled_by_client','2013-10-04');
''', [('2013-10-02', 0.0)])

check("question 4: BOTH kinds of cancellation count", '''
INSERT INTO Users VALUES (1,'No','client'),(10,'No','driver');
INSERT INTO Trips VALUES (1,1,10,1,'cancelled_by_client','2013-10-01'),
                         (2,1,10,1,'cancelled_by_driver','2013-10-01'),
                         (3,1,10,1,'completed','2013-10-01'),
                         (4,1,10,1,'completed','2013-10-01');
''', [('2013-10-01', 0.5)])

check("the boundary dates themselves are included", '''
INSERT INTO Users VALUES (1,'No','client'),(10,'No','driver');
INSERT INTO Trips VALUES (1,1,10,1,'cancelled_by_client','2013-10-01'),
                         (2,1,10,1,'completed','2013-10-03');
''', [('2013-10-01', 1.0), ('2013-10-03', 0.0)])

check("a user who is both a banned client and an unbanned driver id", '''
INSERT INTO Users VALUES (5,'Yes','client'),(6,'No','client'),(10,'No','driver');
INSERT INTO Trips VALUES (1,5,10,1,'completed','2013-10-01'),
                         (2,6,10,1,'cancelled_by_client','2013-10-01');
''', [('2013-10-01', 1.0)])

check("no trips at all", '''
INSERT INTO Users VALUES (1,'No','client');
''', [])

check("both tables empty", '', [])

check("rounding: 2 of 3 cancelled is 0.67", '''
INSERT INTO Users VALUES (1,'No','client'),(10,'No','driver');
INSERT INTO Trips VALUES (1,1,10,1,'cancelled_by_client','2013-10-01'),
                         (2,1,10,1,'cancelled_by_driver','2013-10-01'),
                         (3,1,10,1,'completed','2013-10-01');
''', [('2013-10-01', 0.67)])

## After it passes

- **Check your query against your four sentences**, one at a time, out loud. Then break
  each one on purpose and find the test that catches it: drop the driver join (the banned
  driver case), drop the date filter (the outside-the-window case), change `<> 'completed'`
  to `= 'cancelled_by_client'` (the both-kinds case), and remove the `1.0` (every rate
  becomes `0`). Four requirements, four failures, four named tests.
- **Watch integer division bite.** Run `SELECT 1/3, 1.0/3, ROUND(1.0/3, 2)` with `show`.
  Then decide whether you would rather write `1.0 *` or `CAST(... AS REAL)` - and note
  that MySQL, Postgres and SQLite do not all agree here, which is exactly why being
  explicit is worth the characters.
- **Answer question 6 properly.** Your query cannot produce a day with no unbanned trips,
  because `GROUP BY` has no rows to group. If the executives want every day in the range
  listed with `0.00`, you need a list of dates to `LEFT JOIN` against - a *calendar
  table*. That is how every real reporting system does it, and it is worth knowing the
  name.
- **Then make it the report they actually want.** Add the trip count per day (so a rate of
  `1.00` from a single trip is visibly not a crisis), split cancellations by who cancelled,
  and add `city_id` to the grouping. All three are small changes to the query you have -
  which is a good sign you built the right one.
- Siblings: #1211 Queries Quality and Percentage (conditional aggregation, no joins),
  #1193 Monthly Transactions I, #1174 Immediate Food Delivery II, #185 Department Top
  Three Salaries (the other Hard here).